# Increasing signal to noise ratio via integration gain 

Consider a signal of the form: 

$$
s = A B(t) + n(t)
$$

Where:

n is a gaussian distributed noise with unity variance. 

$ n \sim \mathcal{N}( 0 , 1 ) $

$B(t)$ is a sequence of bits at a specific bit rate $F_b$

A is the amplitude of the signal calculated based on the unity noise variance and a desired carrier to noise density ratio $CN0$


$ A = \sqrt{ 2 \cdot 10^{CN0/10} / F_s } $

Below we will generate this signal and give you the opportunity to change CN0 with a slider. Note that the bits change every bit period so that you can clearly see the edges. 


In [123]:
import numpy as np 
import plotly.graph_objects as go
import ipywidgets as widgets

In [124]:
# Sample rate 
Fs = 1.0e4
# Bit Rate 
Fb = 10
# total time 
T = 1.0
seed = 42



In [125]:
def makeSignal( CN0 ):
    global t, signal
    
    rng = np.random.default_rng(seed=seed)                  # seed is optional, for reproducibility
    L = int( Fs * T )

    # Translate CN0 to variance and std

    var = 1.0 # unity variance
    std = np.sqrt( var )
    noise = rng.normal( loc=0.0, scale=std, size=L)
    # bit signal
    samplesPerBit = int( Fs / Fb )
    nBits = L / samplesPerBit
    bits = np.arange( nBits ) % 2
    symbols = 2*bits-1
    A = np.sqrt( 2 * 10**( CN0 / 10 ) / Fs ) # unity variance -> N0 = 2 / Fs, C = A^2 = CN0 * N0 
    signal = A * np.repeat(symbols, samplesPerBit)

    # Time
    t = np.arange( L ) / Fs



    signal = noise + signal

    # Plot the signal
    # t vs signal
    fig = go.Figure()
    fig.add_trace( go.Scattergl( x=t, y=signal, mode="lines", name="signal" ) )
    fig.update_layout( title=f"Signal - CN0 = {CN0:g} dB-Hz", xaxis_title="t (s)", yaxis_title="amplitude" )
    return fig

# N is a slider for number of samples to integrate over 
def plotCoherent( t , signal, CN0, N ):
    B = len(signal) // N                # number of whole blocks of N samples
    signalClipped = signal[:B*N]
    signalCoherent = np.reshape( signalClipped, (B, N) )
    coherent = np.sum( signalCoherent, axis=1 ) /  N
    tClipped = t[:B*N]
    tCoherent = tClipped[::N] # every Nth sample 

    # plot signal and coherent integration signal so we can see
    # - down sampling 
    # - gain
    fig = go.Figure()
    fig.add_trace( go.Scattergl( x=t, y=signal, mode="lines", name="signal" ) )
    fig.add_trace( go.Scattergl( x=tCoherent, y=coherent, mode="markers", name="coherent" ) )
    fig.update_layout( title=f"Coherent Integration - CN0 = {CN0:g} dB-Hz, N = {N}", xaxis_title="t (s)", yaxis_title="amplitude" )
    return fig

# M is a slider for number of coherent intervals to sum over 
def plotNonCoherent( t, signal, CN0, N, M ):
    # coherent integration over N samples
    B = len(signal) // N                # number of whole blocks of N samples
    signalClipped = signal[:B*N]
    coherent = np.sum( np.reshape( signalClipped, (B, N) ), axis=1 ) / N
    tCoherent = t[:B*N][::N] # every Nth sample 

    # non-coherent integration: square the coherent outputs, then sum M of them
    K = B // M                          # number of whole groups of M coherent outputs
    power = coherent[:K*M]**2
    nonCoherent = np.sum( np.reshape( power, (K, M) ), axis=1 ) / M
    tNonCoherent = tCoherent[:K*M][::M] # every Mth coherent sample

    # plot signal, coherent and non-coherent signal so we can see
    # - squaring removes the bit sign
    # - summing M squared outputs smooths out the noise
    fig = go.Figure()
    fig.add_trace( go.Scattergl( x=t, y=signal, mode="lines", name="signal", opacity=0.3 ) )
    fig.add_trace( go.Scattergl( x=tCoherent, y=coherent, mode="markers", name="coherent" ) )
    fig.add_trace( go.Scattergl( x=tNonCoherent, y=np.sqrt( nonCoherent ), mode="markers", name="non-coherent" ) )
    fig.update_layout( title=f"Non-Coherent Integration - CN0 = {CN0:g} dB-Hz, N = {N}, M = {M}", xaxis_title="t (s)", yaxis_title="amplitude" )
    return fig

In [126]:
# carrier to noise density - dB-Hz 
CN0 = widgets.FloatSlider( value=35, min=0, max=100, step=1, description="CN0 (dB-Hz)", continuous_update=False )

# one plot, redrawn in place when the slider moves
display( CN0 )
# pass plotly's mimebundle directly - display(fig) calls fig.show() and ignores display_id
plot = display( makeSignal( CN0.value )._repr_mimebundle_(), raw=True, display_id=True )
CN0.observe( lambda change: plot.update( makeSignal( change.new )._repr_mimebundle_(), raw=True ), names="value" )

FloatSlider(value=35.0, continuous_update=False, description='CN0 (dB-Hz)', step=1.0)

Notice how as you raise $CN0$ the bits become more and more clear. But for some values like $CN0 = 35$ dB-Hz there are many samples where the bit value is on the wrong side of zero. 

## Coherent Integration

To properly find the bits we want to raise the signal to noise ratio of the signal and we can do this via a very simple digital signal processing method - integration.

To understand why this works we will have to cover some basic math. Recall that:

$$
n \sim \mathcal{N}(0,1)
$$

The discrete signal of interest is:

$$
s_k = B_k A  + n_k 
$$

- $B_k$ can only ever be $\pm 1$ - so we'll simplify that to 1
- $A$ is constant based on CN0 

With this simplification 

$$
s \sim \mathcal{N}(A,1)
$$

The signal to noise ratio of a signal is signal power over noise power. 

$$
SNR = \frac{A^2}{\sigma^2}
$$

Since $\sigma=1$ that means:

$$
SNR_s = \frac{A^2}{1} = A^2 
$$

Now we can introduce integration. Statistics tells us the following rule

If 
$$
a_k \sim \mathcal{N}(A,\sigma^2)
$$

Then 
$$
b = \sum_{k=1}^{N} a_k \sim \mathcal{N}( NA , N\sigma^2 )
$$

And following this we can calculate:

$$
SNR_b = \frac{ (NA)^2 }{ N\sigma^2 } = \frac{ N^2 A^2 }{ N \sigma^2 } = \frac{NA^2}{\sigma^2} = N \cdot SNR_s
$$

In short summation / integration raises the signal to noise ratio of a signal by a factor of $N$ - the number of samples being summed over

This only works because every sample in the sum has the same mean $A$; the signal adds up *in phase*, or **coherently**. If $B_k$ flips sign partway through the sum, the signal terms start cancelling while the noise keeps growing. That's why the slider below only offers values of $N$ that fit evenly inside one bit, and it's the problem the next section deals with.

Note: the plot below shows the **average** ($b/N$) rather than the raw sum, so it stays on the same scale as the original signal. Dividing by $N$ scales signal and noise equally, so the SNR is unchanged: $\mathcal{N}(A, \sigma^2/N)$. Instead of the bits growing, you'll see the noise around them shrink by $\sqrt{N}$.

In [127]:
# coherent integration length - samples
# only divisors of samplesPerBit, so no block of N samples straddles a bit edge
samplesPerBit = int( Fs / Fb )
validN = [ n for n in range( 1, samplesPerBit + 1 ) if samplesPerBit % n == 0 ]
N = widgets.SelectionSlider( options=validN, value=validN[4], description="N (samples)", continuous_update=False )

# one plot, redrawn in place when either slider moves
display( N )
coherentPlot = display( plotCoherent( t, signal, CN0.value, N.value )._repr_mimebundle_(), raw=True, display_id=True )
redrawCoherent = lambda change: coherentPlot.update( plotCoherent( t, signal, CN0.value, N.value )._repr_mimebundle_(), raw=True )
N.observe( redrawCoherent, names="value" )
CN0.observe( redrawCoherent, names="value" )

SelectionSlider(continuous_update=False, description='N (samples)', index=4, options=(1, 2, 4, 5, 8, 10, 20, 2…

## Non-Coherent Integration 

So if we can just sum to raise SNR why don't we just integrate forever. 

There are two reasons:

1. Signals change frequency due to motion (Doppler), and tracking loops need regular updates, so we can't wait forever for a result.
2. More importantly: the pesky data. The bits aren't always 1; they're actually $\pm 1$.

Consider a signal of length 2000 where we no longer assume  $B(t)=1$ but rather 
$$
B_k =
\begin{cases}
1 & \text{for } k < 1000 \\
-1 & \text{for } k \ge 1000
\end{cases}
$$

In this case let's see what happens if we try to integrate for longer than 1000 samples

$$
s_k = A B_k + n_k 
$$

The summed signal $x$

$$
x = \sum_{k=0}^{1999} s_k = \sum_{k=0}^{999}(s_k) + \sum_{k=1000}^{1999}(s_k)
$$


Substituting in $B_k$:

$$
x = \sum_{k=0}^{999}(A + n_k) + \sum_{k=1000}^{1999}(-A + n_k) = \underbrace{1000A - 1000A}_{0} + \sum_{k=0}^{1999} n_k \sim \mathcal{N}(0,\ 2000\sigma^2)
$$

The signal cancels out completely, but the noise doesn't: it keeps accumulating. We summed 2000 samples and ended up with an SNR of **zero**. Even if the bit edge isn't exactly in the middle, every sample on the wrong side of the edge subtracts from the signal while still adding noise.


So what do we do - well if you need to preserve the bits for decoding there isn't much more to do. However, if you want to drive a tracking loop you may not need the bits to do this in which case you actually can integrate across the bit boundaries.

But how. Well if we coherently integrate the signal...then square it...then add the squared signals we can continue to see SNR growth without interference from the bit edges. 

The key is the squaring step: $(+b)^2 = (-b)^2$, so squaring throws away the sign of the bit. Each coherent block (kept inside one bit) comes out near $+NA$ or $-NA$. After squaring, both land near $N^2A^2$, so they add up instead of cancelling. (In a real receiver the signal is complex, and $|I + jQ|^2$ uses the same trick to also throw away the unknown carrier phase.)

So now let's recall that 

- $N$ is the number of samples coherently integrated - we will try to maximize this at the bit length
- $M$ is the number of non-coherent integrations

So let's do the math and see what happens

If:

$$
s_k \sim \mathcal{N}(A , \sigma^2 )
$$

then the coherent sum is also a normal distribution 

$$
b_j = \sum_{k=1}^{N} s_k \sim \mathcal{N}( NA, N\sigma^2 )
$$

but now we introduce the square of this distribution. The square of a normal distribution is $\chi^2$

$$
c_j = \left| b_j \right|^2 \sim \chi^2(  N^2 A^2 + N \sigma^2 , 4 N^3 A^2 \sigma^2 + 2 N^2 \sigma^4 )
$$

Note: strictly, $\chi^2$ is parameterized by degrees of freedom and noncentrality, not mean and variance. Here (and for $d$ below) we write it as $\chi^2(\text{mean}, \text{variance})$ because SNR only needs those two moments. They come from $E[X^2] = \mu^2 + v$ and $\text{Var}(X^2) = 4\mu^2 v + 2v^2$ for $X \sim \mathcal{N}(\mu, v)$, with $\mu = NA$ and $v = N\sigma^2$.

The mean of $c_j$ includes $N\sigma^2$ even when there's no signal at all. That's noise power that squaring turned into a positive offset. Only the $N^2A^2$ part comes from the signal, so:

$$
SNR_{c} = \frac{ (N^2 A^2)^2 }{4 N^3 A^2 \sigma^2 + 2 N^2 \sigma^4} = \frac{ N^4 A^4 }{ 2N^2\sigma^2(2NA^2+\sigma^2)} = \frac{ N^2 A^4 }{ 2\sigma^2(2NA^2+\sigma^2)}
$$

And finally by summing 

$$
d = \sum_{j=1}^{M} c_j \sim \chi^2 \left( MN^2A^2 + MN\sigma^2 , 4 M N^3 A^2 \sigma^2 + 2 M N^2 \sigma^4 \right)
$$

Then 

$$
SNR_d = M \cdot SNR_c = M \frac{ N^2 A^4 }{ 2\sigma^2(2NA^2+\sigma^2)}
$$


Writing this in terms of the coherent SNR, $SNR_b = NA^2/\sigma^2$, makes the trade-off clear:

$$
SNR_d = M \frac{SNR_b^2}{2(2\,SNR_b + 1)}
$$

$SNR_d$ grows linearly with $M$, so non-coherent integration lets us keep gaining past the bit edge, where $b_j$ couldn't. But it's less efficient than coherent integration. When $SNR_b$ is small, $SNR_d \approx M \cdot SNR_b^2/2$: doubling $M$ doubles it, but doubling $N$ quadruples it. This is called **squaring loss**, and it's why we make $N$ as long as possible (up to the bit length) before adding non-coherent intervals.

To demonstrate this use the slider below to raise non-coherent integration factor. You'll notice that **the bits are destroyed** but that the amount of noise on the remaining signal goes down as $M$ goes up.

Note: the non-coherent markers show $\sqrt{d/M}$ so they're in the same amplitude units as the other traces. They sit just above $|A|$ (at $\sqrt{A^2 + \sigma^2/N}$) because of the noise offset from squaring, the same term we left out of $SNR_c$ above.

In [128]:
# non-coherent integration length - number of coherent intervals
M = widgets.IntSlider( value=1, min=1, max=10, step=1, description="M (intervals)", continuous_update=False )

# one plot, redrawn in place when any slider moves
display( M )
nonCoherentPlot = display( plotNonCoherent( t, signal, CN0.value, N.value, M.value )._repr_mimebundle_(), raw=True, display_id=True )
redrawNonCoherent = lambda change: nonCoherentPlot.update( plotNonCoherent( t, signal, CN0.value, N.value, M.value )._repr_mimebundle_(), raw=True )
M.observe( redrawNonCoherent, names="value" )
N.observe( redrawNonCoherent, names="value" )
CN0.observe( redrawNonCoherent, names="value" )

IntSlider(value=1, continuous_update=False, description='M (intervals)', max=10, min=1)

## Probability of detection and false alarm


Now that we've discussed how SNR can be raised by integrating both coherently and non-coherently lets discuss what this means if we are trying to do detection.

We'll define the following terms

**Detection:** Accurately indicating that a signal is present in the environemnt 
**False Alarm:** Incorrectly indicating that a signal is present in the environment when there is only noise 

Lets design a very simple detection algorithm based on signal strength $S$ and detection threshold $D$

$$
\begin{cases}
\text{Detection}, & S > D,\\
\text{No detection}, & S \le D.
\end{cases}
$$

This algorithm is simple but instructive. 


So lets now mathematically define **Probablility of detection** as the probaility that the signal exists and we detect it. We can do this by integrating the probability distribution of the signal itself over the interval $\left[ D, \infty\right]$


$$
P_D = \int_D^{\infty} f_S(s \mid \text{Present})\, ds
$$

Where $ f_S(s \mid \text{Present}  $ is the probability distribution of the measured environment when signal is present.

We will also introduce the equation for **Probability of False Alarm** as 

$$
P_{FA} = \int_0^D f_S( s \mid \text{Not-present})\, ds 
$$

So lets examine our four cases 

$S_0 = n_k$ - noise only 
$S_1 = a_k$ - signal plus noise no processing
$S_2 = b_j$ - signal plus noise plus coherent integration 
$S_3 = d_i$ - signal plus noise plus coherent and non coherent integration 

Below we will plot the probability distribution of all four cases for the values of CN0, N and M selected and show you how the distribution lays relative to the detection threshold of choice.

Below you can play with CN0, N, M and D. What you should be able to find is that coherent and non-coherent integration provide significant seperation of the probability distributions between $S_0$ and $S_2,S_3$. In a real device you want low $P_{FA}$ while maintaining high $P_D$

In [129]:
from plotly.subplots import make_subplots
from scipy import stats

# Three plots in a subplot, each comparing noise only (S0) against signal present,
# with S0 put through the same processing as the signal
# no processing:   n ~ N(0, 1)                        vs  a ~ N(A, 1)
# coherent:        sum of N n's ~ N(0, N)             vs  b ~ N(NA, N)
# non-coherent:    d with no signal ~ N * chi2( M )   vs  d ~ N * noncentral chi2( M dof, nc = M*N*A^2 )
# each plot has its own threshold, D1 D2 D3, in noise standard deviations above that plot's noise mean
def plotDetection( CN0, N, M, D1, D2, D3 ):
    A = np.sqrt( 2 * 10**( CN0 / 10 ) / Fs )

    cases = [
        ( "S0 vs S1: no processing", stats.norm( 0, 1 ), stats.norm( A, 1 ), D1 ),
        ( f"S0 vs S2: coherent, N = {N}", stats.norm( 0, np.sqrt( N ) ), stats.norm( N*A, np.sqrt( N ) ), D2 ),
        # b_j / sqrt(N) ~ N(sqrt(N)A, 1), so d / N = sum of M of those squared ~ ncx2( M, M*N*A^2 )
        ( f"S0 vs S3: non-coherent, N = {N}, M = {M}", stats.chi2( df=M, scale=N ), stats.ncx2( df=M, nc=M*N*A**2, scale=N ), D3 ),
    ]

    titles = []
    plots = []
    for name, noise, present, D in cases:
        threshold = noise.mean() + D * noise.std()
        PFA = noise.sf( threshold )
        PD = present.sf( threshold )
        # cover both distributions and make sure the threshold is always in view
        lo = min( noise.ppf( 1e-6 ), present.ppf( 1e-6 ), threshold )
        hi = max( noise.ppf( 1 - 1e-6 ), present.ppf( 1 - 1e-6 ), threshold )
        pad = 0.05 * ( hi - lo )
        x = np.linspace( lo - pad, hi + pad, 2001 )
        titles.append( f"{name}, D = {D:g} noise std<br>P_D = {PD:.4f}, P_FA = {PFA:.4f}" )
        plots.append( ( x, noise.pdf( x ), present.pdf( x ), threshold ) )

    fig = make_subplots( rows=3, cols=1, subplot_titles=titles, vertical_spacing=0.12 )
    for i, ( x, yNoise, yPresent, threshold ) in enumerate( plots ):
        row = i + 1
        first = i == 0                  # only put each trace in the legend once
        above = x >= threshold
        fig.add_trace( go.Scatter( x=x, y=yNoise, mode="lines", line=dict( color="#d62728" ), name="noise only", legendgroup="noise", showlegend=first ), row=row, col=1 )
        fig.add_trace( go.Scatter( x=x, y=yPresent, mode="lines", line=dict( color="#1f77b4" ), name="signal present", legendgroup="present", showlegend=first ), row=row, col=1 )
        # shade the area above the threshold - these are the integrals that give P_FA and P_D
        fig.add_trace( go.Scatter( x=x[above], y=yNoise[above], mode="lines", fill="tozeroy", line=dict( color="#d62728" ), name="P_FA", legendgroup="PFA", showlegend=first ), row=row, col=1 )
        fig.add_trace( go.Scatter( x=x[above], y=yPresent[above], mode="lines", fill="tozeroy", line=dict( color="#1f77b4" ), name="P_D", legendgroup="PD", showlegend=first ), row=row, col=1 )
        fig.add_vline( x=threshold, line=dict( color="black", dash="dash" ), row=row, col=1 )
        fig.update_xaxes( title_text="amplitude", row=row, col=1 )
        fig.update_yaxes( title_text="pdf", row=row, col=1 )
    fig.update_layout( title=f"Detection - CN0 = {CN0:g} dB-Hz", height=1000 )
    return fig

# detection threshold per plot - noise standard deviations above the noise mean
D1 = widgets.FloatSlider( value=2, min=0, max=5, step=0.1, description="D1 (S1)", continuous_update=False )
D2 = widgets.FloatSlider( value=2, min=0, max=5, step=0.1, description="D2 (S2)", continuous_update=False )
D3 = widgets.FloatSlider( value=2, min=0, max=5, step=0.1, description="D3 (S3)", continuous_update=False )

# one plot, redrawn in place when any slider moves
display( D1, D2, D3 )
detectionPlot = display( plotDetection( CN0.value, N.value, M.value, D1.value, D2.value, D3.value )._repr_mimebundle_(), raw=True, display_id=True )
redrawDetection = lambda change: detectionPlot.update( plotDetection( CN0.value, N.value, M.value, D1.value, D2.value, D3.value )._repr_mimebundle_(), raw=True )
D1.observe( redrawDetection, names="value" )
D2.observe( redrawDetection, names="value" )
D3.observe( redrawDetection, names="value" )
M.observe( redrawDetection, names="value" )
N.observe( redrawDetection, names="value" )
CN0.observe( redrawDetection, names="value" )

FloatSlider(value=0.5, continuous_update=False, description='D (threshold)', max=3.0, step=0.05)